<a href="https://colab.research.google.com/github/safwanwaris/EduOS-Memory-Manager/blob/main/MemoryMgr_EduOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
EduOS Memory Manager Simulation
A Lightweight OS for Smart Classrooms

This script simulates a paging system and demonstrates three
standard page replacement algorithms: FIFO, LRU, and Optimal.

Reference String: [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2, 1, 2, 0, 1, 7, 0, 1]
Number of Frames: 3
"""

from typing import List, Tuple


def validate_inputs(reference_string: List[int], num_frames: int) -> None:
    """Validates simulation inputs before running any algorithm."""
    if not reference_string:
        raise ValueError("Reference string cannot be empty.")
    if num_frames <= 0:
        raise ValueError("Number of frames must be a positive integer.")


def display_steps(
    algorithm_name: str,
    steps: List[Tuple[int, List[int], bool]],
    total_faults: int
) -> None:
    """Prints a formatted table of simulation steps and the total page fault count."""
    print(f"\n--- {algorithm_name} Page Replacement ---")
    print(f"{'Page Request':<15} | {'Frames State':<25} | {'Status'}")
    print("-" * 60)
    for page, frames, fault in steps:
        status = "Page Fault" if fault else "Hit"
        print(f"{page:<15} | {str(frames):<25} | {status}")
    print("-" * 60)
    print(f"Total Page Faults: {total_faults}\n")


def fifo_page_replacement(
    reference_string: List[int], num_frames: int
) -> Tuple[int, List[Tuple[int, List[int], bool]]]:
    """
    Simulates the First-In, First-Out (FIFO) page replacement algorithm.

    Maintains a queue of pages in the order they were loaded. When a
    replacement is needed, the oldest loaded page (front of the queue)
    is evicted.

    Args:
        reference_string: Ordered list of page numbers to access.
        num_frames: Maximum number of frames available in physical memory.

    Returns:
        A tuple of (total_page_faults, steps) where steps is a list of
        (page, frames_snapshot, was_fault) tuples.
    """
    frames: List[int] = []
    page_faults = 0
    steps: List[Tuple[int, List[int], bool]] = []

    for page in reference_string:
        fault = False
        if page not in frames:
            page_faults += 1
            fault = True
            if len(frames) == num_frames:
                frames.pop(0)  # Evict oldest page (FIFO order)
            frames.append(page)

        steps.append((page, list(frames), fault))

    return page_faults, steps


def lru_page_replacement(
    reference_string: List[int], num_frames: int
) -> Tuple[int, List[Tuple[int, List[int], bool]]]:
    """
    Simulates the Least Recently Used (LRU) page replacement algorithm.

    Tracks the last-accessed index for each page currently in frames.
    When eviction is needed, the page with the smallest (oldest) last-use
    index is replaced.

    Note: Only pages currently in frames are candidates for eviction.
    Pages not yet loaded are not included in the recent_usage lookup.

    Args:
        reference_string: Ordered list of page numbers to access.
        num_frames: Maximum number of frames available in physical memory.

    Returns:
        A tuple of (total_page_faults, steps).
    """
    frames: List[int] = []
    page_faults = 0
    steps: List[Tuple[int, List[int], bool]] = []
    recent_usage: dict[int, int] = {}  # page -> last accessed index

    for i, page in enumerate(reference_string):
        fault = False
        if page not in frames:
            page_faults += 1
            fault = True

            if len(frames) == num_frames:
                # Evict the page in frames with the oldest (smallest) last-use index.
                # All candidates are guaranteed to be in recent_usage because they
                # were recorded when first loaded.
                lru_page = min(frames, key=lambda p: recent_usage[p])
                frames.remove(lru_page)
            frames.append(page)

        # Update last-use timestamp for the accessed page (hit or fault)
        recent_usage[page] = i

        steps.append((page, list(frames), fault))

    return page_faults, steps


def optimal_page_replacement(
    reference_string: List[int], num_frames: int
) -> Tuple[int, List[Tuple[int, List[int], bool]]]:
    """
    Simulates the Optimal (OPT / Belady's) page replacement algorithm.

    For each eviction decision, looks ahead in the reference string to
    find the page currently in memory whose next use is farthest in the
    future (or never). That page is evicted.

    This algorithm is theoretically perfect but unimplementable in a real
    OS (future references are unknown). It serves as a lower-bound
    benchmark for page fault counts.

    Args:
        reference_string: Ordered list of page numbers to access.
        num_frames: Maximum number of frames available in physical memory.

    Returns:
        A tuple of (total_page_faults, steps).
    """
    frames: List[int] = []
    page_faults = 0
    steps: List[Tuple[int, List[int], bool]] = []

    for i, page in enumerate(reference_string):
        fault = False
        if page not in frames:
            page_faults += 1
            fault = True

            if len(frames) == num_frames:
                farthest_use = -1
                page_to_replace = frames[0]  # Safe default

                for candidate in frames:
                    try:
                        # Next occurrence of this page after current position
                        next_use = reference_string.index(candidate, i + 1)
                    except ValueError:
                        # Page never used again — ideal eviction candidate
                        next_use = float('inf')  # type: ignore[assignment]

                    if next_use > farthest_use:
                        farthest_use = next_use
                        page_to_replace = candidate

                frames.remove(page_to_replace)
            frames.append(page)

        steps.append((page, list(frames), fault))

    return page_faults, steps


def print_summary(fifo_faults: int, lru_faults: int, opt_faults: int) -> None:
    """Prints a final comparative summary of all three algorithms."""
    print("===========================================")
    print("              FINAL SUMMARY                ")
    print("===========================================")
    print(f"{'Algorithm':<12} | {'Page Faults':<12} | {'vs Optimal'}")
    print("-" * 45)
    print(f"{'FIFO':<12} | {fifo_faults:<12} | +{fifo_faults - opt_faults} extra faults")
    print(f"{'LRU':<12} | {lru_faults:<12} | +{lru_faults - opt_faults} extra faults")
    print(f"{'Optimal':<12} | {opt_faults:<12} | Baseline (theoretical)")
    print("-" * 45)

    best_practical = "LRU" if lru_faults <= fifo_faults else "FIFO"
    print(f"\nBest practical algorithm: {best_practical}")
    print(
        "\nKey Insight:\n"
        "  Optimal gives the theoretical minimum page faults, since it has\n"
        "  perfect knowledge of future references — impossible in a real OS.\n"
        "  LRU exploits temporal locality (recently used pages are likely\n"
        "  needed again), making it the recommended choice for EduOS where\n"
        "  students repeatedly toggle between a small set of active apps."
    )


if __name__ == "__main__":
    print("===========================================")
    print("   EduOS Memory Manager Simulation Start   ")
    print("===========================================")

    # --- Input Setup ---
    # Represents a student alternating between classroom apps
    # (virtual lab, PDF reader, browser, quiz tool, etc.)
    reference_string: List[int] = [
        7, 0, 1, 2, 0, 3, 0, 4, 2, 3,
        0, 3, 2, 1, 2, 0, 1, 7, 0, 1
    ]
    num_frames: int = 3  # Budget classroom device: 3 memory frames

    try:
        validate_inputs(reference_string, num_frames)
    except ValueError as e:
        print(f"[Input Error] {e}")
        raise SystemExit(1)

    print(f"Reference String : {reference_string}")
    print(f"Available Frames : {num_frames}\n")

    # --- Run Simulations ---
    fifo_faults, fifo_steps = fifo_page_replacement(reference_string, num_frames)
    display_steps("FIFO", fifo_steps, fifo_faults)

    lru_faults, lru_steps = lru_page_replacement(reference_string, num_frames)
    display_steps("LRU", lru_steps, lru_faults)

    opt_faults, opt_steps = optimal_page_replacement(reference_string, num_frames)
    display_steps("Optimal", opt_steps, opt_faults)

    # --- Summary ---
    print_summary(fifo_faults, lru_faults, opt_faults)

   EduOS Memory Manager Simulation Start   
Reference String : [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2, 1, 2, 0, 1, 7, 0, 1]
Available Frames : 3


--- FIFO Page Replacement ---
Page Request    | Frames State              | Status
------------------------------------------------------------
7               | [7]                       | Page Fault
0               | [7, 0]                    | Page Fault
1               | [7, 0, 1]                 | Page Fault
2               | [0, 1, 2]                 | Page Fault
0               | [0, 1, 2]                 | Hit
3               | [1, 2, 3]                 | Page Fault
0               | [2, 3, 0]                 | Page Fault
4               | [3, 0, 4]                 | Page Fault
2               | [0, 4, 2]                 | Page Fault
3               | [4, 2, 3]                 | Page Fault
0               | [2, 3, 0]                 | Page Fault
3               | [2, 3, 0]                 | Hit
2               | [2, 3, 0]            